# Activity 1: RAGAS Evaluation with Cost Analysis

Compare a Fireworks AI (`gpt-oss-20b`) RAG pipeline against an OpenAI (`gpt-4.1-mini`) RAG pipeline using RAGAS metrics and LangSmith cost tracing.

## Cell 1: Imports & Environment Setup

In [32]:
import os
from dotenv import load_dotenv
load_dotenv()

# LangChain / LangSmith
from langchain_openai import ChatOpenAI
from langchain_fireworks import ChatFireworks
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore

# RAGAS
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision  # PascalCase
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


print("All imports successful!")

All imports successful!


/var/folders/c3/gpyjtk951l19x3vj98kvm25r0000gn/T/ipykernel_65477/3766455917.py:17: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision  # PascalCase
/var/folders/c3/gpyjtk951l19x3vj98kvm25r0000gn/T/ipykernel_65477/3766455917.py:17: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision  # PascalCase
/var/folders/c3/gpyjtk951l19x3vj98kvm25r0000gn/T/ipykernel_65477/3766455917.py:17: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metric

## Cell 2: Load & Chunk the PDF

Load `data/cat-health-guide.pdf`, split it into chunks, and build an in-memory Qdrant vector store retriever backed by Fireworks embeddings.

**Hint:** Look at `app/rag.py` — the `_build_rag_graph()` function does exactly this. Adapt that logic here.

In [33]:
# Token length function (same pattern as rag.py)
def tiktoken_len(text: str) -> int:
    tokens = tiktoken.encoding_for_model("gpt-4o").encode(text)
    return len(tokens)

# Load the PDF
loader = PyMuPDFLoader("data/cat-health-guide.pdf")
documents = loader.load()
print(f"Loaded {len(documents)} pages from PDF")

# Split documents into chunks (token-aware, chunk_size=750)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,
    chunk_overlap=0,
    length_function=tiktoken_len,
)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

# Fireworks embeddings (OpenAI-compatible interface pointed at Fireworks base URL)
embedding_model = OpenAIEmbeddings(
    model="accounts/fireworks/models/qwen3-embedding-8b",
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base="https://api.fireworks.ai/inference/v1",
    check_embedding_ctx_length=False,
)

# Build in-memory Qdrant vector store from chunks + embeddings
qdrant_vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embedding_model,
    location=":memory:",
    collection_name="cat_health_eval",
)

# Create retriever
retriever = qdrant_vectorstore.as_retriever()

print(f"Loaded and chunked documents. Retriever ready.")

Loaded 22 pages from PDF
Split into 42 chunks
Loaded and chunked documents. Retriever ready.


## Cell 3: Define the RAG Prompt & Two LLMs

Both pipelines share the same prompt and retriever — only the LLM changes.

In [34]:
human_template = (
    "#CONTEXT:\n{context}\n\nQUERY:\n{query}\n\n"
    "Use the provided context to answer the query. "
    "If the answer is not in the context, say \"I don't know\"."
)
prompt = ChatPromptTemplate.from_messages([("human", human_template)])

# Fireworks LLM (uses FIREWORKS_API_KEY from env automatically)
fireworks_llm = ChatFireworks(
    model="accounts/fireworks/models/gpt-oss-20b",
)

# OpenAI LLM (uses OPENAI_API_KEY from env automatically)
openai_llm = ChatOpenAI(
    model="gpt-4.1-mini",
)

# Build two chains: same prompt, different LLM, same output parser
fireworks_chain = prompt | fireworks_llm | StrOutputParser()
openai_chain = prompt | openai_llm | StrOutputParser()

print("Chains ready!")

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x11c21f9d0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x11c21fd90>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x11c21e0d0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x11c21d090>


Chains ready!


## Cell 4: Define Test Questions & Ground Truths

Write 5 questions about cat health that your PDF can answer, plus a short expected answer for each.

In [35]:
questions = [
    "What are the core vaccinations recommended for kittens?",
    "What are the defined feline life stages and their age ranges?",
    "What are signs of degenerative joint disease in senior cats?",
    "What is the recommended litter box setup for a multicat household?",
    "What are the nutritional recommendations for senior cats over 10 years old?",
]

ground_truths = [
    "Core vaccinations for kittens include rabies virus, feline herpesvirus type 1 (FHV-1), feline calicivirus (FCV), and feline panleukopenia virus (FPV). Feline leukemia virus (FeLV) vaccination is also considered core for kittens due to age-related susceptibility.",
    "The four age-related feline life stages are: Kitten (birth up to 1 year), Young Adult (1 through 6 years), Mature Adult (7 to 10 years), and Senior (over 10 years). End of life is a fifth stage that can occur at any age.",
    "Signs of degenerative joint disease in senior cats include changes or reduction in jumping and climbing behavior, reduced muscle mass, and behavioral changes. Studies suggest 40-92% of all cats may present with clinical signs associated with DJD, and it is more prevalent and severe in older cats.",
    "The recommendation is one litter box per cat plus one additional box, placed in multiple quiet locations throughout the house that are easily accessible and provide escape routes. The box should be at least 1.5 times the length of the cat from nose to tail tip.",
    "Senior cats over 10 years may need their resting energy requirements (RER) multiplied by a factor of 10-25%. They should not be protein restricted and need a minimum protein allowance of 30-45% dry matter. Being underweight is common in senior cats due to reduced digestive capabilities.",
]

print(f"Defined {len(questions)} test questions with ground truths.")

Defined 5 test questions with ground truths.


## Cell 5: Run Both Pipelines & Collect Results

For each question, retrieve context then run both LLMs. Collect results into two lists of dicts for RAGAS.

Each dict needs: `question`, `answer`, `contexts` (list of strings), `ground_truth`.

In [36]:
fireworks_results = []
openai_results = []

for question, ground_truth in zip(questions, ground_truths):
    # Retrieve relevant docs (shared retriever for both chains)
    docs = retriever.invoke(question)

    # Format docs into a context string to pass to the chain
    context = "\n\n".join([doc.page_content for doc in docs])

    # Run both chains with the same context
    fireworks_answer = fireworks_chain.invoke({"query": question, "context": context})
    openai_answer = openai_chain.invoke({"query": question, "context": context})

    # Contexts must be a list of strings (one per retrieved doc chunk)
    context_list = [doc.page_content for doc in docs]

    fireworks_results.append({
        "question": question,
        "answer": fireworks_answer,
        "contexts": context_list,
        "ground_truth": ground_truth,
    })

    openai_results.append({
        "question": question,
        "answer": openai_answer,
        "contexts": context_list,
        "ground_truth": ground_truth,
    })

    print(f"Q: {question[:60]}... ✓")

print("\nAll questions processed!")

Q: What are the core vaccinations recommended for kittens?... ✓
Q: What are the defined feline life stages and their age ranges... ✓
Q: What are signs of degenerative joint disease in senior cats?... ✓
Q: What is the recommended litter box setup for a multicat hous... ✓
Q: What are the nutritional recommendations for senior cats ove... ✓

All questions processed!


## Cell 6: RAGAS Evaluation

Convert your results lists into `datasets.Dataset` objects and run `evaluate()`.

In [37]:
# Wrap LangChain objects for RAGAS compatibility
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

# Initialize metrics
metrics = [
    Faithfulness(llm=ragas_llm),
    AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings),
    ContextPrecision(llm=ragas_llm),
]

# Build datasets
fireworks_dataset = Dataset.from_list(fireworks_results)
openai_dataset = Dataset.from_list(openai_results)

print("Evaluating Fireworks pipeline...")
fireworks_eval = evaluate(
    dataset=fireworks_dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("Evaluating OpenAI pipeline...")
openai_eval = evaluate(
    dataset=openai_dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("Evaluation complete!")

/var/folders/c3/gpyjtk951l19x3vj98kvm25r0000gn/T/ipykernel_65477/3323574425.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
/var/folders/c3/gpyjtk951l19x3vj98kvm25r0000gn/T/ipykernel_65477/3323574425.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


Evaluating Fireworks pipeline...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating OpenAI pipeline...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluation complete!


## Cell 7: Compare Results

In [38]:
import pandas as pd

fireworks_df = fireworks_eval.to_pandas()
openai_df = openai_eval.to_pandas()

print("=== Fireworks (gpt-oss-20b) ===")
print(fireworks_df.mean(numeric_only=True))

print("\n=== OpenAI (gpt-4.1-mini) ===")
print(openai_df.mean(numeric_only=True))

# Side-by-side comparison
print("\n=== Side-by-Side Comparison ===")
comparison = pd.DataFrame({
    "Fireworks (gpt-oss-20b)": fireworks_df.mean(numeric_only=True),
    "OpenAI (gpt-4.1-mini)": openai_df.mean(numeric_only=True),
})
print(comparison)

=== Fireworks (gpt-oss-20b) ===
faithfulness         0.702121
answer_relevancy     0.927310
context_precision    0.900000
dtype: float64

=== OpenAI (gpt-4.1-mini) ===
faithfulness         1.000000
answer_relevancy     0.962003
context_precision    0.916667
dtype: float64

=== Side-by-Side Comparison ===
                   Fireworks (gpt-oss-20b)  OpenAI (gpt-4.1-mini)
faithfulness                      0.702121               1.000000
answer_relevancy                  0.927310               0.962003
context_precision                 0.900000               0.916667


## Cell 8: LangSmith Cost Analysis

After running cells 5-7, go to [smith.langchain.com](https://smith.langchain.com) and check the **session-16-eval** project.

Look at the **Cost** column for each trace to compare per-query cost between Fireworks and OpenAI.

Record your findings below:

### Cost Comparison Notes

| Metric | Fireworks gpt-oss-20b | OpenAI gpt-4.1-mini |
|--------|----------------------|---------------------|
| Avg cost per query | ~2-3s latency | ~7-10s latency |
| faithfulness | 0.702 | 1.000 |
| answer_relevancy | 0.927 | 0.962 |
| context_precision | 0.900 | 0.917 |

**Analysis:** OpenAI gpt-4.1-mini significantly outperformed gpt-oss-20b on 
faithfulness (1.0 vs 0.70), indicating it stays grounded in retrieved context 
and hallucinates far less. Answer relevancy and context precision were 
comparable across both providers. Fireworks delivered notably lower latency 
(~2-5s vs ~7-10s per query), demonstrating a clear speed-quality trade-off: 
the open-source model responds faster but is less faithful to the provided 
context, while gpt-4.1-mini is slower but produces more reliable, 
context-grounded answers.